# Агент "Куда сходить в Алматы"

ТЗ #2: парсинг sxodim.com/almaty + агент, отвечающий на вопросы о том, куда сходить.

## Раздел 1: Парсинг сайта

Скрейпинг запускается один раз и кэшируется в `sxodim_raw.json` — при повторном запуске ноутбука сеть не дёргается, если файл уже существует. Логика парсинга живёт в `scraper.py` (см. рядом), ноутбук только вызывает её и показывает результат.

In [1]:
import json
from pathlib import Path

import requests

import scraper

RAW_PATH = Path("sxodim_raw.json")

In [2]:
if RAW_PATH.exists():
    records = json.loads(RAW_PATH.read_text(encoding="utf-8"))
    print(f"Loaded {len(records)} cached records from {RAW_PATH}")
else:
    session = requests.Session()
    fetch = scraper.make_fetcher(session)
    records = scraper.scrape_all(fetch)
    scraper.save_raw(records, str(RAW_PATH))
    print(f"Scraped {len(records)} records -> {RAW_PATH}")

Scraped 595 records -> sxodim_raw.json


In [3]:
events = [r for r in records if r["type"] == "event"]
places = [r for r in records if r["type"] == "place"]
print(f"events: {len(events)}, places: {len(places)}, total: {len(records)}")

events: 139, places: 456, total: 595


In [4]:
print("Пример события:")
print(json.dumps(events[0], ensure_ascii=False, indent=2))
print()
print("Пример места:")
print(json.dumps(places[0], ensure_ascii=False, indent=2))

Пример события:
{
  "type": "event",
  "url": "https://sxodim.com/almaty/event/abzal-teshovty-koncerti",
  "name": "Абзал Утешов Алматыда",
  "category": "Концерты во Дворце Республики",
  "description": "Абзал Утешов – қазақстандық эстраданың жас әрі жарқын өкілдерінің бірі. Ол тыңдармандардың жүрегінен орын алған әсерлі әндері мен шынайы орындау мәнері арқылы кеңінен танымал болды.",
  "address": "Республика сарайы, Достық даңғылы, 56",
  "date_start": "2026-10-07T20:00:00+05:00",
  "price": 11000,
  "currency": "KZT",
  "rating": null,
  "image": "https://sxodim.com/uploads/posts/2026/08/02/optimized/d0fcc3bb75454a3f7c4f79e31ae74adb_1522x570-q-85.jpg"
}

Пример места:
{
  "type": "place",
  "url": "https://sxodim.com/almaty/place/17st-lounge-bar",
  "name": "17st. Lounge Bar",
  "category": "Бары",
  "description": "Посетители смогут окунуться в атмосферу вкусных стейков, крафтового пенного и дымных сетов.",
  "address": "17st. Lounge Bar, ул. Розыбакиева, 166",
  "date_start": null

Дальше (следующий этап, отдельным пайплайном): нормализация в `sxodim_data.json` с семантическими тегами и сам агент — см. `docs/superpowers/specs/2026-09-28-sxodim-agent-design.md`.